# Possession Lens — latest local notebook
Authors: Hrishikesh Karmalkar, Basel, Omar, Michail and Paul.

Run from top to bottom. The default run reads public external aggregate reports and runs local checks. Training, import and competition inference require explicit switches. No competition data, video, labels or predictions are uploaded. Clear all outputs before sharing.

The target is **possession-spell F1**. The default production checkpoint remains `rnn-boundary-001.npz` until a defensible promotion decision. A higher score on partial provider labels alone does not establish challenge accuracy.


In [ ]:
from pathlib import Path
import sys, json, subprocess, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "controlswitch").is_dir()), None)
assert ROOT is not None, "Open this notebook within the Possession Lens folder."
sys.path.insert(0, str(ROOT))
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
import numpy as np
import pandas as pd
from IPython.display import display, FileLink
print("Project:", ROOT.name, "| Python:", sys.version.split()[0])
MODEL = ROOT / "models/rnn-boundary-001.npz"
EUROPE_ROOT = ROOT / "data/external/skillcorner-europe/raw"
EUROPE_REPORT = ROOT / "reports/external/rnn-europe-001-report.json"
RUN_IMPORT = False
RUN_TRAINING = False
RUN_INFERENCE = False
RUN_TESTS = True


## 1. Five European matches
Training: Manchester City–Liverpool (4039), Dortmund–Bayern (3749), Juventus–Inter (3518).
Validation: PSG–Marseille (2269). Reserved test: Real Madrid–Barcelona (3442).

Source: official SkillCorner open data, MIT licence, pinned revision `1ed50b775f93075c2a6d00d0d795f13b7b89709e`. Preserve attribution. Inputs are ignored by Git. Provider-generated home/away possession is weak supervision; null is **unknown**, never dead ball. Adjacent observed tracking is interpolated from 10 Hz to the common 25 Hz interface; gaps are not filled.

Do not open test labels during development. The experiment freezes the checkpoint and decoder before test loading. After the first result, this match is a viewed test; rerunning it reproduces evidence and cannot create another fresh test.


In [ ]:
from controlswitch.skillcorner_legacy import SPLIT
print("Frozen split:", SPLIT)
manifest_path = EUROPE_ROOT / "import-manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print("Verified imported files:", len(manifest["files"]))
else:
    print("External archive not imported locally yet; source-only checkouts do not include raw data.")


## 2. Import the already downloaded archive (optional)
Download the pinned official repository ZIP in your browser. This cell extracts only the five selected matches, verifies expected file sizes and Git blob hashes, and preserves the licence. It does not parse test possession labels. Existing destinations are protected from overwrite.


In [ ]:
ARCHIVE = Path.home() / "Downloads/opendata-1ed50b775f93075c2a6d00d0d795f13b7b89709e.zip"
if RUN_IMPORT:
    from controlswitch.skillcorner_legacy import import_archive
    catalog = json.loads((ROOT / "docs/external/top-five-europe-catalog.json").read_text())
    imported = import_archive(ARCHIVE, catalog, EUROPE_ROOT)
    print("Verified files:", len(imported["files"]))
else:
    print("Import disabled; existing local data remains available.")


## 3. Latest five-league experiment
A new candidate is trained **from scratch** on the three selected top-five-league matches. One- and two-layer RNNs use 24 units per layer, dropout, L2 and gradient clipping. Up to 12 sampled iterations are compared. Checkpoints and decoder penalties are selected by mean clean/dropout **validation spell F1**, rather than training loss. Training/validation loss remains an overfitting diagnostic.

All models receive the same tracking and label mask. No dead-ball targets are invented. Reported F1 is conditional on provider-labelled runs, not independently annotated full-match possession.


In [ ]:
if EUROPE_REPORT.exists():
    europe = json.loads(EUROPE_REPORT.read_text())
    rows = []
    for name, result in europe["test_results"].items():
        rows.append({"model": name, "clean_spell_F1": result["clean"]["spell_f1_mean"],
                     "dropout_spell_F1": result["masked"]["spell_f1_mean"],
                     "mean_spell_F1": result["selection_spell_f1"]})
    display(pd.DataFrame(rows).round(4))
    print("Candidate beats current on this conditional test:", europe["candidate_beats_current"])
    print("Promoted to default:", europe["promoted_to_default"])
    print("Selected configuration:", europe["frozen"])
    display(pd.DataFrame(europe["test_audit"]))
    for limitation in europe["limitations"]:
        print("•", limitation)
else:
    print("European experiment report is not available yet. No test score is assumed.")


In [ ]:
learning_path = ROOT / "reports/external/rnn-europe-001-validation-trials.json"
if learning_path.exists():
    learning = pd.DataFrame(json.loads(learning_path.read_text()))
    best_per_iteration = learning.sort_values("score", ascending=False).drop_duplicates(["layers", "iteration"]).sort_values(["layers", "iteration"])
    display(best_per_iteration.round(4))
    print("A sampled iteration uses at most 1,024 windows of 128 frames; it is not a full epoch.")


## Latest: recovering missed and merged possessions

The latest experiment keeps the original three training matches and PSG–Marseille validation. Previously scored Real Madrid–Barcelona is **excluded from development**. A different match, Bayern–Dortmund on 9 November 2019 (2417), was reserved before training and opened only after freezing the selected model.

The candidate uses spell-balanced loss to emphasise short possessions, plus an optional trained boundary head to reduce switching costs where trajectory/geometry supports a change. Unknown-censored spells are not treated as complete short possessions. Unknown frames carry zero loss; no dead-ball labels are invented. Model selection maximises validation spell F1 subject to false-positive and short-pulse limits. Extra layers and more iterations are not assumed to improve F1.

The fresh test shares clubs with training. Provider labels cover only observed H/A runs, so results remain conditional transfer evidence, not full-match challenge accuracy. Once scored, 2417 is a viewed test and must not guide further tuning.


In [ ]:
RECOVERY_REPORT = ROOT / "reports/external/rnn-europe-recovery-001-report.json"
if RECOVERY_REPORT.exists():
    recovery = json.loads(RECOVERY_REPORT.read_text())
    display(pd.DataFrame([{"model": name,
        "clean_spell_F1": r["clean"]["spell_f1_mean"],
        "dropout_spell_F1": r["masked"]["spell_f1_mean"],
        "mean_spell_F1": r["selection_spell_f1"]}
        for name, r in recovery["test_results"].items()]).round(4))
    print("Test: Bayern–Dortmund, Bundesliga, 9 November 2019")
    print("Selected configuration:", recovery["frozen"])
    print("Recovery changes accepted:", recovery["recovery_changes_accepted"])
    print(recovery["decision"])
    print("Candidate beats previous European RNN:", recovery["candidate_beats_previous_european"])
    print("Promoted to production default:", recovery["promoted_to_default"])
    diagnostic_rows = []
    for model, conditions in recovery["test_diagnostics"].items():
        for condition, d in conditions.items():
            diagnostic_rows.append({"model": model, "condition": condition,
                "missed_spells_IoU_0.5": d["missed_reference_spells_at_iou_0_5"],
                "false_positive_spells_IoU_0.5": d["false_positive_spells_at_iou_0_5"],
                "merge_overlap_count": d["predictions_overlapping_multiple_reference_spells"],
                "pulses_under_0.4s": d["predicted_spells_under_0_4_seconds"]})
    display(pd.DataFrame(diagnostic_rows))
    print("Merge overlap counts are descriptive geometry, not confirmed event annotations.")
    display(pd.DataFrame(recovery["test_audit"]))
    for limitation in recovery["limitations"]:
        print("•", limitation)
else:
    print("Recovery experiment report not available yet. No improvement is assumed.")


In [ ]:
recovery_learning = ROOT / "reports/external/rnn-europe-recovery-001-learning.json"
if recovery_learning.exists():
    display(pd.DataFrame(json.loads(recovery_learning.read_text())).round(4))
    print("Training objective is weighted; train/validation probe CE is unweighted for comparison.")
    print("Checkpoint selection uses validation spell F1 with false-positive limits, not minimum training loss.")


In [ ]:
IMPORT_FRESH_TEST = False
if IMPORT_FRESH_TEST:
    from controlswitch.skillcorner_fresh_import import import_fresh_test
    imported_fresh = import_fresh_test(ARCHIVE, EUROPE_ROOT)
    print("Reserved match imported without parsing tracking:", len(imported_fresh["files"]), "verified files")
else:
    print("Sixth-match import disabled. Enable only if 2417 is not already imported.")


### Optional: reproduce the recovery experiment
Leave disabled for the judge walkthrough. This is a lengthy local retraining run. Reproduction uses the already viewed test and cannot provide new independent evidence. It does not replace the production checkpoint. All raw files must already be imported locally.


In [ ]:
RUN_RECOVERY_TRAINING = False
if RUN_RECOVERY_TRAINING:
    from datetime import datetime
    from controlswitch.europe_recovery import run as recover_spells
    recovery_out = ROOT / "local" / ("recovery-reproduction-" + datetime.now().strftime("%Y%m%d-%H%M%S"))
    recovery_out.mkdir(parents=True)
    protocol = json.loads((ROOT / "reports/external/rnn-europe-recovery-001-preregistered-protocol.json").read_text())
    protocol["test_previously_parsed"] = True
    protocol["execution_mode"] = "reproduction_on_viewed_test"
    (recovery_out / "preregistered-protocol.json").write_text(json.dumps(protocol, indent=2) + "\n")
    recover_spells(EUROPE_ROOT, recovery_out,
        ROOT / "models/rnn-europe-001-experimental.npz", MODEL, iterations=12)
else:
    print("Recovery retraining disabled. Existing results are shown above.")


## 4. Reproduce external-only training (optional, potentially lengthy)
Leave disabled for the judge walkthrough. It creates a new local experiment and does not replace the default checkpoint. The held-out result becomes reproduction of an already viewed test. Never reuse it for model selection or tune after reading the result.


In [ ]:
if RUN_TRAINING:
    from datetime import datetime
    from controlswitch.skillcorner_europe import run
    reproduction_out = ROOT / "local" / ("europe-reproduction-" + datetime.now().strftime("%Y%m%d-%H%M%S"))
    run(EUROPE_ROOT, reproduction_out, MODEL, iterations=12)
else:
    print("Training disabled. Default checkpoint:", MODEL.name)


## 5. Earlier independent test evidence
The current model was retained after the J03WR9 test. These historical second-division results remain comparison evidence; they are excluded from the new European first-division training pool. Historical test matches are already viewed and cannot be reused as fresh tests.


In [ ]:
historical_path = ROOT / "reports/external/rnn-fresh-J03WR9-001-report.json"
if historical_path.exists():
    historical = json.loads(historical_path.read_text())
    display(pd.DataFrame([{"model": name,
        "clean_spell_F1": r["clean"]["spell_f1_mean"],
        "dropout_spell_F1": r["masked"]["spell_f1_mean"],
        "mean_spell_F1": r["mean_clean_masked_spell_f1"]}
        for name, r in historical["fresh_match_results"].items()]).round(4))


## 6. Judge experience and player statistics
Open `dashboard.html` from the project folder, or use `Open Possession Lens.command`. It starts the **named external Bochum–Leverkusen example**. The Match, Players and Model evidence tabs show team possession, provider-mapped names and model evidence. Challenge tracking slots have no permitted name mapping and remain provisional tracklets; they cannot support verified individual-player claims.

Team percentages exclude dead and unknown frames. Individual control shares must preserve unassigned time and are distinct from total team possession. No manual challenge player mapping or substitution annotations are permitted.


In [ ]:
if (ROOT / "dashboard.html").exists():
    display(FileLink(str(ROOT / "dashboard.html")))
print("Named external demo by default; private challenge outputs remain local.")


## 7. Apply the frozen model to the challenge locally (optional)
This runs inference only, with no training, tuning, calibration or video comparison. All outputs remain in the ignored `local` directory. No private statistics are displayed by this notebook. Do not publish notebook outputs or private match files.


In [ ]:
DATA_ROOT = ROOT / "data/raw/provided"
if RUN_INFERENCE:
    from datetime import datetime
    prediction_out = ROOT / "local" / ("notebook-inference-" + datetime.now().strftime("%Y%m%d-%H%M%S"))
    subprocess.run([sys.executable, "-m", "possession_lens", "rnn-predict", "--data-root", str(DATA_ROOT), "--model", str(MODEL), "--out", str(prediction_out)], cwd=ROOT, check=True)
    print("Local inference completed. Keep its results private.")
else:
    print("Challenge inference disabled.")


## 8. Local checks and sharing
Share source, the cleared notebook, dependencies, checkpoint, licence notices and aggregate external reports. Never upload challenge data, video, manually generated labels, prediction timelines, local manifests or raw external assets by accident. The source archive excludes `data`, `local`, `work` and `.git`.


In [ ]:
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-q"], cwd=ROOT, check=True)
print("Notebook complete. Clear all outputs before committing or sharing.")
